# ⚙️ Yêu cầu cài đặt (Prerequisites)
Trước khi chạy, hãy đảm bảo máy tính của bạn có:
Python 3.x: Đã cài đặt Python (Khuyên dùng bản 3.9 trở lên).
Trình duyệt: Đã cài đặt Google Chrome hoặc Microsoft Edge (Tool sẽ tự động tìm).
Thư viện Python: Mở Terminal/Command Prompt và chạy lệnh sau:
```bash
pip install DrissionPage pandas
```

#🚀 Hướng dẫn Sử dụng
## Bước 1: Tạo file code

1. Bấm nút ▶️ (Run) ở ô Code bên dưới (Cell có chữ `%%writefile main.py`).

2. Chờ 1 giây để hệ thống đóng gói code.

In [ ]:
%%writefile main.py

from DrissionPage import ChromiumPage, ChromiumOptions
import pandas as pd
import time
import re
from datetime import datetime

# --- CẤU HÌNH ---
BASE_URL = "https://batdongsan.com.vn/ban-nha-dat-tp-hcm"
PARAM = "?vrs=1"
TRANG_BAT_DAU = 1
# Số lượng tab mở cùng lúc
BATCH_SIZE = 20


def lay_tong_so_trang(page):
    max_page = 1
    try:
        if page.wait.ele_displayed('.re__pagination-group', timeout=5):
            page_numbers = page.eles('.re__pagination-number')
            ds_trang = [int(item.text.strip()) for item in page_numbers if item.text.strip().isdigit()]
            if ds_trang: max_page = max(ds_trang)
    except:
        pass
    return max_page


def get_links(page):
    links = []
    try:
        elems = page.eles('tag:a@@href:pr')
        for ele in elems:
            link = ele.link
            if link and 'batdongsan.com.vn' in link and re.search(r'pr\d+', link):
                links.append(link)
    except:
        pass
    return list(set(links))


def xu_ly_dia_chi(full_address):
    if not full_address: return "", "", ""
    parts = [p.strip() for p in full_address.split(',')]
    if parts and ("Hồ Chí Minh" in parts[-1] or "HCM" in parts[-1]): parts.pop()
    Quan_Huyen = parts.pop() if parts else ""
    Phuong_Xa = parts.pop() if parts else ""
    Duong = ", ".join(parts) if parts else ""
    return Duong, Phuong_Xa, Quan_Huyen


def extract_data_from_tab(tab, url):
    data = {
        "Mã tin": "", "Ngày đăng": "",
        "Đường": "", "Xã/phường/thị trấn": "", "Huyện/quận": "",
        "Khoảng giá": "", "Diện tích": "",
        "Số phòng ngủ": "", "Số phòng tắm/vệ sinh": "", "Số tầng": "",
        "Hướng nhà": "", "Hướng ban công": "", "Mặt tiền": "",
        "Đường vào": "", "Pháp lý": "", "Nội thất": "",
        "URL": url
    }

    try:
        # Chờ H1 xuất hiện (Dấu hiệu trang đã load xong nội dung chính)
        if not tab.wait.ele_displayed('tag:h1', timeout=5):
            return None

        # 1. TIÊU ĐỀ & ĐỊA CHỈ
        h1 = tab.ele('tag:h1')

        raw_address = ""
        # Tìm địa chỉ bằng class trước
        addr_ele = tab.ele('.js__pr-address', timeout=1)
        if addr_ele:
            raw_address = addr_ele.text
        # Nếu không thấy, dùng fallback H1
        elif h1 and h1.next():
            raw_address = h1.next().text

        if raw_address:
            d, p, q = xu_ly_dia_chi(raw_address)
            data["Đường"] = d
            data["Xã/phường/thị trấn"] = p
            data["Huyện/quận"] = q

        # 2. THÔNG SỐ (Dùng s_eles quét html tĩnh)
        specs = tab.s_eles('.re__pr-specs-content-item')
        for item in specs:
            label, value = "", ""
            title_ele = item.s_ele('.re__pr-specs-content-item-title')
            val_ele = item.s_ele('.re__pr-specs-content-item-value')
            if title_ele and val_ele:
                label = title_ele.text.strip()
                value = val_ele.text.strip()

            if "Mức giá" in label or "Khoảng giá" in label:
                data["Khoảng giá"] = value
            elif "Diện tích" in label:
                data["Diện tích"] = value
            elif "phòng ngủ" in label:
                data["Số phòng ngủ"] = value
            elif "tắm" in label or "vệ sinh" in label:
                data["Số phòng tắm/vệ sinh"] = value
            elif "tầng" in label:
                data["Số tầng"] = value
            elif "Hướng nhà" in label:
                data["Hướng nhà"] = value
            elif "Hướng ban công" in label:
                data["Hướng ban công"] = value
            elif "Mặt tiền" in label:
                data["Mặt tiền"] = value
            elif "Đường vào" in label:
                data["Đường vào"] = value
            elif "Pháp lý" in label:
                data["Pháp lý"] = value
            elif "Nội thất" in label:
                data["Nội thất"] = value

        # 3. MÃ TIN & NGÀY ĐĂNG
        date_label = tab.s_ele('text:Ngày đăng')
        if date_label and date_label.next(): data["Ngày đăng"] = date_label.next().text.strip()

        code_label = tab.s_ele('text:Mã tin')
        if code_label and code_label.next(): data["Mã tin"] = code_label.next().text.strip()

        if not data["Mã tin"]:
            match = re.search(r'pr(\d+)', url)
            if match: data["Mã tin"] = match.group(1)

    except Exception:
        return None

    return data


def main():
    print("Đang khởi động chương trình...")

    co = ChromiumOptions()
    co.auto_port()
    # Tắt load ảnh để tăng tốc độ load web
    co.set_argument('--blink-settings=imagesEnabled=false')

    page = ChromiumPage(addr_or_opts=co)

    all_data = []
    columns = [
        "Mã tin", "Ngày đăng", "Đường", "Xã/phường/thị trấn", "Huyện/quận",
        "Khoảng giá", "Diện tích", "Số phòng ngủ", "Số phòng tắm/vệ sinh",
        "Số tầng", "Hướng nhà", "Hướng ban công", "Mặt tiền",
        "Đường vào", "Pháp lý", "Nội thất", "URL"
    ]

    # --- LẤY TỔNG SỐ TRANG ---
    first_url = f"{BASE_URL}{PARAM}"
    print(f"\n--- Đang truy cập trang chủ để quét phân trang: {first_url} ---")
    page.get(first_url)
    tong_so_trang = lay_tong_so_trang(page)
    print(f"Đã tìm thấy: {tong_so_trang} trang.")

    TRANG_KET_THUC = tong_so_trang
    print(f"Số trang: {TRANG_BAT_DAU} -> {TRANG_KET_THUC}")

    for i in range(TRANG_BAT_DAU, TRANG_KET_THUC + 1):
        url = f"{BASE_URL}{PARAM}" if i == 1 else f"{BASE_URL}/p{i}{PARAM}"
        print(f"\n--- Trang {i}/{TRANG_KET_THUC}: {url}")

        try:
            if i != 1: page.get(url)

            if not page.wait.ele_displayed('tag:a@@href:pr', timeout=10):
                print("Không thấy danh sách tin. Bỏ qua trang này.")
                continue

            links = get_links(page)
            print(f"   -> Tìm thấy {len(links)} link.")

            # --- XỬ LÝ THEO TỪNG ĐỢT (BATCH) ---
            for j in range(0, len(links), BATCH_SIZE):
                batch_links = links[j: j + BATCH_SIZE]
                if j // BATCH_SIZE + 1 == 1:
                    print(f"   Đang xử lý {len(batch_links)} tab...")
                else:
                    print(f"   Đợt {j // BATCH_SIZE + 1}: Đang xử lý {len(batch_links)} tab...")

                opened_tabs = []

                # 1. Mở tab
                for link in batch_links:
                    try:
                        tab = page.new_tab(link)
                        opened_tabs.append({"tab": tab, "url": link})
                    except:
                        pass

                # 2. Chờ Load (3 giây)
                time.sleep(3)

                # 3. Quét
                for item in opened_tabs:
                    tab = item["tab"]
                    link = item["url"]
                    try:
                        info = extract_data_from_tab(tab, link)
                        if info and (info.get('Khoảng giá') or info.get('Diện tích')):
                            all_data.append(info)
                    except:
                        pass
                    finally:
                        tab.close()  # Đóng ngay sau khi quét xong

            print(f"   Đã quét xong trang {i}.")
            # Lưu tạm sau mỗi trang
            if len(all_data) > 0 and i % 2 == 0:
                pd.DataFrame(all_data)[columns].to_csv("ket_qua_temp.csv", index=False, encoding='utf-8-sig')

        except Exception as e:
            print(f"Lỗi xử lý trang {i}: {e}")

    # --- LƯU KẾT QUẢ CUỐI CÙNG ---
    if len(all_data) > 0:
        now = datetime.now()
        time_str = now.strftime("%d_%m_%Y %I_%M_%S %p")
        file_name = f"gia_nha_hcm ({time_str}).csv"

        pd.DataFrame(all_data)[columns].to_csv(file_name, index=False, encoding='utf-8-sig')
        print(f"\nHoàn thành! Kết quả lưu tại: {file_name}")
        print(f"\nTổng cộng: {len(all_data)} tin.")
        print(f"\nBạn có thể đóng trình duyệt")
    else:
        print("\nKhông thu thập được tin nào.")


if __name__ == "__main__":
    main()

## Bước 2: Tải file về

Sau khi chạy xong bước 1, bấm nút ▶️ (Run) ở ô Code bên dưới để tải file main.py về máy tính.

In [ ]:
# @title 📥 Bấm vào đây để tải tool về máy
from google.colab import files
try:
    files.download('main.py')
    print("✅ Đang tải file main.py về máy tính của bạn...")
except Exception as e:
    print("⚠️ Lỗi: Trình duyệt chặn popup. Hãy kiểm tra icon trên thanh địa chỉ.")

## Bước 3: Chạy trên máy của bạn
1. Mở thư mục chứa file `main.py` vừa tải.

2. Mở Terminal (hoặc CMD) tại thư mục đó và gõ lệnh:
```Bash
python main.py
```
3. Quan sát:
* Trình duyệt sẽ tự động bật lên. Đừng tắt nó.
* Bạn sẽ thấy tool mở hàng loạt tab (Batch) rồi tự đóng lại liên tục.
* Terminal sẽ hiển thị tiến độ: Đã quét xong trang X...

# 📊 Kết quả đầu ra
Sau khi chạy xong, file `.csv` sẽ được tạo tại thư mục hiện tại với tên định dạng: `gia_nha_hcm (<ngày>_<tháng>_<năm> <giờ>_<phút>_<giây>).csv`.


# 📋 Mô tả bộ dữ liệu:
1. Mã tin: Mã số định danh duy nhất (ID) của bài đăng do hệ thống https://www.batdongsan.com.vn cấp.

2. Ngày đăng: Ngày tháng mà chủ nhà hoặc môi giới đăng tải (hoặc làm mới) tin này lên website.

3. Đường: Tên đường, phố hoặc số nhà cụ thể của bất động sản.

4. Xã/phường/thị trấn: Tên đơn vị hành chính cấp xã/phường nơi có bất động sản.

5. Huyện/quận: Tên đơn vị hành chính cấp quận/huyện.

6. Khoảng giá: Mức giá rao bán do người đăng cung cấp.

7. Diện tích: Diện tích đất hoặc diện tích sử dụng được ghi trong tin đăng.

8. Số phòng ngủ: Tổng số lượng phòng ngủ trong nhà.

9. Số phòng tắm/vệ sinh: Tổng số lượng nhà vệ sinh (WC).

10. Số tầng: Số lượng tầng lầu của ngôi nhà (bao gồm trệt, lầu, thượng...).

11. Hướng nhà: Hướng của cửa chính ngôi nhà (Đông, Tây, Nam, Bắc...).

12. Hướng ban công: Hướng nhìn ra của ban công (thường áp dụng cho chung cư hoặc nhà cao tầng).

13. Mặt tiền: Chiều rộng của cạnh nhà tiếp giáp với đường đi.

14. Đường vào: Độ rộng của con đường hoặc hẻm trước nhà.

15. Pháp lý: Tình trạng giấy tờ sở hữu của bất động sản.

16. Nội thất: Thông tin về đồ đạc, trang thiết bị để lại cho người mua.

17. URL: Đường dẫn trực tiếp đến bài đăng gốc trên website.

In [ ]:
import pandas as pd
df = pd.read_csv('gia_nha_hcm (18_12_2025 02_01_37 AM).csv')
df

,Mã tin,Ngày đăng,Đường,Xã/phường/thị trấn,Huyện/quận,Khoảng giá,Diện tích,Số phòng ngủ,Số phòng tắm/vệ sinh,Số tầng,Hướng nhà,Hướng ban công,Mặt tiền,Đường vào,Pháp lý,Nội thất,URL
0,44707861,13/12/2025,Đường Phạm Văn Chiêu,Phường 9,Gò Vấp,"6,6 tỷ","60,9 m²",2 phòng,2 phòng,1 tầng,NaN,NaN,"4,35 m",6 m,Sổ đỏ/ Sổ hồng,Cơ bản,https://batdongsan.com.vn/ban-nha-rieng-duong-...
1,44062690,17/12/2025,230/38/8 Đường Phan Văn Trị,Phường 12,Bình Thạnh,"10,4 tỷ",58 m²,9 phòng,10 phòng,6 tầng,Đông - Bắc,NaN,NaN,NaN,Sổ đỏ/ Sổ hồng,Đầy đủ,https://batdongsan.com.vn/ban-nha-rieng-duong-...
2,44730477,16/12/2025,Đường Nguyễn Văn Khạ,Thị trấn Củ Chi,Củ Chi,"3,5 tỷ",100 m²,2 phòng,2 phòng,1 tầng,NaN,NaN,5 m,3 m,Sổ đỏ/ Sổ hồng,Không nội thất,https://batdongsan.com.vn/ban-nha-rieng-duong-...
3,44810236,16/12/2025,271 Đường Hoàng Hoa Thám,Phường 13,Tân Bình,"17,9 tỷ","57,8 m²",NaN,NaN,1 tầng,NaN,NaN,"3,8 m",22 m,Sổ đỏ/ Sổ hồng,NaN,https://batdongsan.com.vn/ban-nha-mat-pho-duon...
4,44812217,16/12/2025,"Dự án PiCity High Park, Đường Thạnh Xuân 13",Phường Thạnh Xuân,Quận 12,"6,7 tỷ","107,5 m²",1 phòng,1 phòng,2 tầng,NaN,NaN,NaN,NaN,Sổ đỏ/ Sổ hồng,Không nội thất,https://batdongsan.com.vn/ban-shophouse-nha-ph...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2961,44681834,16/12/2025,Đường Nguyễn Thị Thập,Phường Tân Phú,Quận 7,55 tỷ,165 m²,2 phòng,2 phòng,2 tầng,Nam,Nam,6 m,30 m,Sổ đỏ/ Sổ hồng,Cơ bản,https://batdongsan.com.vn/ban-nha-mat-pho-duon...
2962,44801783,15/12/2025,Đường 3B,Phường Bình Hưng Hòa A,Bình Tân,"2,7 tỷ",18 m²,2 phòng,2 phòng,3 tầng,Đông,Đông,3 m,10 m,Sổ đỏ/ Sổ hồng,Không nội thất,https://batdongsan.com.vn/ban-nha-mat-pho-duon...
2963,44740431,08/12/2025,Đường Nguyễn Văn Hưởng,Phường Thảo Điền,Thủ Đức,70 tỷ,200 m²,3 phòng,4 phòng,3 tầng,Đông - Bắc,Đông - Bắc,10 m,10 m,Sổ đỏ/ Sổ hồng,Đầy đủ.,https://batdongsan.com.vn/ban-nha-rieng-duong-...
2964,42742297,15/12/2025,Đường Dương Cát Lợi,Thị trấn Nhà Bè,Nhà Bè,"5,99 tỷ","38,8 m²",2 phòng,2 phòng,2 tầng,NaN,NaN,"3,2 m",12 m,Sổ hồng,Đầy đủ,https://batdongsan.com.vn/ban-nha-mat-pho-duon...
